# Team Season - common_team_roster

This notebook inspects the data **downloaded** from the `common_team_roster` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/common_team_roster`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [63]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "common_team_roster"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [64]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


Endpoint: common_team_roster
Folder: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/01_raw/2025_26/02_team_season/common_team_roster
Parquet files: 30


,file,size_mb
0,common_team_roster__team_id=1610612737.parquet,0.018
1,common_team_roster__team_id=1610612738.parquet,0.018
2,common_team_roster__team_id=1610612739.parquet,0.018
3,common_team_roster__team_id=1610612740.parquet,0.018
4,common_team_roster__team_id=1610612741.parquet,0.018
5,common_team_roster__team_id=1610612742.parquet,0.018
6,common_team_roster__team_id=1610612743.parquet,0.018
7,common_team_roster__team_id=1610612744.parquet,0.018
8,common_team_roster__team_id=1610612745.parquet,0.018
9,common_team_roster__team_id=1610612746.parquet,0.018


In [65]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


,file,rows,cols
0,common_team_roster__team_id=1610612737.parquet,30,28
1,common_team_roster__team_id=1610612738.parquet,23,28
2,common_team_roster__team_id=1610612739.parquet,28,28
3,common_team_roster__team_id=1610612740.parquet,24,28
4,common_team_roster__team_id=1610612741.parquet,26,28
5,common_team_roster__team_id=1610612742.parquet,30,28
6,common_team_roster__team_id=1610612743.parquet,24,28
7,common_team_roster__team_id=1610612744.parquet,30,28
8,common_team_roster__team_id=1610612745.parquet,26,28
9,common_team_roster__team_id=1610612746.parquet,30,28


In [66]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


Combined shape: (819, 28)
Total rows: 819
Total columns: 28


In [67]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,819,28,22932,8854,38.61


In [68]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
SORT_SEQUENCE,761,92.918
SUB_SORT_SEQUENCE,526,64.225
COACH_TYPE,526,64.225
IS_ASSISTANT,526,64.225
COACH_NAME,526,64.225
LAST_NAME,526,64.225
FIRST_NAME,526,64.225
COACH_ID,526,64.225
SCHOOL,299,36.508
NUM,298,36.386


In [69]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",0,0.000
"(0.01, 0.05]",0,0.000
"(0.05, 0.1]",0,0.000
"(0.1, 0.25]",0,0.000
"(0.25, 0.5]",526,64.225
"(0.5, 0.75]",293,35.775
"(0.75, 1.0]",0,0.000


In [70]:
if df.empty:
    col_dist = pd.DataFrame()
else:
    col_null_pct = df.isna().mean()
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    col_bins = pd.cut(col_null_pct, bins=bins, include_lowest=True)
    col_counts = col_bins.value_counts().sort_index()
    col_dist = pd.DataFrame({
        "columns": col_counts,
        "pct_columns": (col_counts / len(col_null_pct) * 100).round(3),
    })

col_dist


,columns,pct_columns
"(-0.001, 0.01]",5,17.857
"(0.01, 0.05]",0,0.000
"(0.05, 0.1]",0,0.000
"(0.1, 0.25]",0,0.000
"(0.25, 0.5]",15,53.571
"(0.5, 0.75]",7,25.000
"(0.75, 1.0]",1,3.571


In [71]:
# Merge all teams, filter CommonTeamRoster, drop all-null columns, and export

files = sorted(data_dir.glob("*.parquet"))
dfs = [pd.read_parquet(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if df_all.empty:
    print("No data to filter.")
else:
    table_col = "TABLE_NAME" if "TABLE_NAME" in df_all.columns else None

    if table_col is None:
        print("Required column not found: TABLE_NAME")
        print(df_all.columns)
    else:
        before_rows = len(df_all)
        df_filtered = df_all.loc[df_all[table_col] == "CommonTeamRoster"].copy()

        # Drop columns that are entirely null after filtering
        all_null_cols = [c for c in df_filtered.columns if df_filtered[c].isna().all()]
        if all_null_cols:
            df_filtered = df_filtered.drop(columns=all_null_cols)

        after_rows = len(df_filtered)

        print(f"Rows before: {before_rows}")
        print(f"Rows after: {after_rows}")
        print(f"Removed: {before_rows - after_rows}")
        print(f"Dropped all-null columns: {all_null_cols}")

        out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "02_team_season"
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / "common_team_roster.parquet"
        df_filtered.to_parquet(out_path, index=False)
        print(f"Saved to: {out_path}")


Rows before: 819
Rows after: 526
Removed: 293
Dropped all-null columns: ['COACH_ID', 'FIRST_NAME', 'LAST_NAME', 'COACH_NAME', 'IS_ASSISTANT', 'COACH_TYPE', 'SORT_SEQUENCE', 'SUB_SORT_SEQUENCE']
Saved to: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/common_team_roster.parquet
